# xsmith walkthrough

One target. One pass through the pipeline. Requires `ANTHROPIC_API_KEY`.


In [2]:
import sys, os, pathlib, asyncio
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

# env = ROOT / ".env"
# if env.exists():
#     for line in env.read_text().splitlines():
#         if "=" in line and not line.strip().startswith("#"):
#             k, v = line.split("=", 1)
#             os.environ.setdefault(k.strip(), v.strip())

# assert os.environ.get("ANTHROPIC_API_KEY"), "set ANTHROPIC_API_KEY"


## 1. Target module


In [ ]:
from xsmith.domain.target import Target

SOURCE = '''
def classify(n):
    if n < 0:
        return "negative"
    if n == 0:
        return "zero"
    if n % 2 == 0:
        return "positive even"
    return "positive odd"
'''.lstrip()

target = Target(target_id="demo/classify", module_path="demo.classify", source=SOURCE)


## 2. Enumerate goal universe

Run an import-only test under coverage.py to learn every branch arc in the module (the universe of goals for the coverage instance).

In [ ]:
from xsmith.execution.subprocess import SubprocessEvaluator

evaluator = SubprocessEvaluator(timeout_s=15.0)
universe = await evaluator.enumerate_goals(target)
target.goals = universe
print(len(universe), "goals")
for g in sorted(universe, key=lambda g: (g.src, g.dst)):
    print(" ", g.key())

## 3. Initial progress state

In [ ]:
from xsmith.domain.progress import Progress

progress = Progress(all=universe)
print(f"{len(progress.hit)}/{len(progress.all)} hit")

## 4. One generator call (variant 1 = edge cases)


In [ ]:
from xsmith.agents.generator import GeneratorAgent

gen = GeneratorAgent(variant_idx=1, model="claude-sonnet-4-6", max_turns=6)
candidate, gen_usage = await gen.propose(target=target, missing=progress.missing, history=[])

print("rationale:", candidate.rationale)
print("---")
print(candidate.code)
print(f"cost: ${gen_usage.cost_usd:.4f}, in={gen_usage.tokens_in}, out={gen_usage.tokens_out}")

## 5. One scorer call on that candidate


In [ ]:
from xsmith.agents.scorer import ScorerAgent

scorer = ScorerAgent(model="claude-sonnet-4-6", max_turns=3, gamma=0.5)
score, score_usage = await scorer.score(target=target, missing=progress.missing, candidate=candidate)
print(f"Q = {score.immediate_goals} + {score.gamma} * {score.future_value} = {score.q}")

## 6. Evaluate the candidate, update progress

In [ ]:
evaluation = await evaluator.evaluate(candidate, target)
new = progress.update(evaluation.goals_hit)
print(f"outcome={evaluation.outcome}, +{len(new)} new goals, hit={len(progress.hit)}/{len(progress.all)}")
if evaluation.stderr.strip():
    print("stderr:", evaluation.stderr[:400])

## 7. Full strategy step (K=5 parallel generators + scoring + argmax)

This is what one step of `Explorer` does internally.

In [ ]:
from xsmith.strategies.qvalue import QValueStrategy

strategy = QValueStrategy(model="claude-sonnet-4-6", k=5, gamma=0.5)
winner, usage = await strategy.propose(target=target, progress=progress, history=[])
print(winner.rationale)
print("---")
print(winner.code)
print(f"aggregate cost: ${usage.cost_usd:.4f}")

## 8. Drive the loop for a small budget


In [ ]:
from xsmith.domain.budget import Budget
from xsmith.exploration.explorer import Explorer

progress = Progress(all=universe)  # reset
budget = Budget(steps=3)
explorer = Explorer(strategy=strategy, evaluator=evaluator)
result = await explorer.run(target=target, budget=budget, initial_progress=progress)

for step in result.steps:
    print(f"step {step.iteration}: {step.evaluation.outcome:>5}  +{len(step.new_goals)} new  hit={step.hit_after}/{step.total}  ${step.agent_usage.cost_usd:.4f}")
print(f"final: {result.hit_count}/{result.total_count} ({result.fraction:.0%})")